In [4]:
from google.colab import drive
import os

drive.mount('/content/drive')

PROJECT_ROOT = "/content/drive/MyDrive/Colab Notebooks/Porfolio/Balearia/2 Demanda"

# Creamos estructura del proyecto
os.makedirs(f"{PROJECT_ROOT}/data/raw", exist_ok=True)
os.makedirs(f"{PROJECT_ROOT}/data/processed", exist_ok=True)
os.makedirs(f"{PROJECT_ROOT}/reports/figures", exist_ok=True)
os.makedirs(f"{PROJECT_ROOT}/reports/model_cards", exist_ok=True)
os.makedirs(f"{PROJECT_ROOT}/dashboards/powerbi_mockups", exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Exists?:", os.path.exists(PROJECT_ROOT))
print("Processed folder:", os.path.exists(f"{PROJECT_ROOT}/data/processed"))

Mounted at /content/drive
PROJECT_ROOT: /content/drive/MyDrive/Colab Notebooks/Porfolio/Balearia/2 Demanda
Exists?: True
Processed folder: True


In [5]:
import pandas as pd
import numpy as np

trips_df = pd.read_parquet(f"{PROJECT_ROOT}/data/processed/trips.parquet")
bookings_df = pd.read_parquet(f"{PROJECT_ROOT}/data/processed/bookings.parquet")

trips_df["departure_datetime_local"] = pd.to_datetime(trips_df["departure_datetime_local"])
trips_df["date"] = pd.to_datetime(trips_df["date"])

trips_df = trips_df.sort_values(["route_id","departure_datetime_local"]).reset_index(drop=True)

# Checks
assert trips_df["trip_id"].is_unique
assert trips_df["pax_real"].ge(0).all()
assert (trips_df["pax_real"] <= trips_df["capacity_pax"]).all()
print("Quality checks OK ✅")


def build_feature_store(trips):
    df = trips.copy()
    df = df.sort_values(["route_id", "departure_datetime_local"]).reset_index(drop=True)

    df["pax_lag_1w"] = df.groupby("route_id")["pax_real"].shift(7)
    df["pax_lag_2w"] = df.groupby("route_id")["pax_real"].shift(14)
    df["veh_lag_1w"] = df.groupby("route_id")["veh_real"].shift(7)
    df["veh_lag_2w"] = df.groupby("route_id")["veh_real"].shift(14)

    gp = df.groupby("route_id")["pax_real"]
    df["pax_roll_mean_20"] = gp.shift(1).rolling(20).mean().reset_index(level=0, drop=True)
    df["pax_roll_std_20"]  = gp.shift(1).rolling(20).std().reset_index(level=0, drop=True)

    gv = df.groupby("route_id")["veh_real"]
    df["veh_roll_mean_20"] = gv.shift(1).rolling(20).mean().reset_index(level=0, drop=True)
    df["veh_roll_std_20"]  = gv.shift(1).rolling(20).std().reset_index(level=0, drop=True)

    gd = df.groupby("route_id")["delay_minutes"]
    df["delay_roll_mean_20"] = gd.shift(1).rolling(20).mean().reset_index(level=0, drop=True)

    df["occ_pax_roll_mean_20"] = (df["pax_roll_mean_20"] / df["capacity_pax"]).clip(0, 1)

    return df

feature_store = build_feature_store(trips_df)
feature_store = feature_store.dropna(subset=["pax_lag_1w","pax_roll_mean_20"]).reset_index(drop=True)

feature_store.to_parquet(f"{PROJECT_ROOT}/data/processed/trip_feature_store.parquet", index=False)
print("Saved ✅ data/processed/trip_feature_store.parquet")

display(feature_store.head())

Quality checks OK ✅
Saved ✅ data/processed/trip_feature_store.parquet


,company,trip_id,route_id,origin_port,dest_port,departure_datetime_local,date,dep_time,weekday,month,...,pax_lag_1w,pax_lag_2w,veh_lag_1w,veh_lag_2w,pax_roll_mean_20,pax_roll_std_20,veh_roll_mean_20,veh_roll_std_20,delay_roll_mean_20,occ_pax_roll_mean_20
0,LevanteFerries,T0000075,DEN-FOR,Denia,Formentera,2024-01-07 08:00:00,2024-01-07,08:00,6,1,...,542.0,533.0,194.0,160.0,581.25,157.994295,173.80,44.684390,5.60,0.645833
1,LevanteFerries,T0000076,DEN-FOR,Denia,Formentera,2024-01-07 12:00:00,2024-01-07,12:00,6,1,...,631.0,528.0,214.0,162.0,587.10,156.085638,175.95,43.950301,5.60,0.652333
2,LevanteFerries,T0000077,DEN-FOR,Denia,Formentera,2024-01-07 17:00:00,2024-01-07,17:00,6,1,...,591.0,399.0,168.0,114.0,585.10,155.248494,174.35,43.221918,6.40,0.650111
3,LevanteFerries,T0000078,DEN-FOR,Denia,Formentera,2024-01-07 20:00:00,2024-01-07,20:00,6,1,...,733.0,393.0,192.0,142.0,588.70,155.762674,173.30,43.510555,5.75,0.654111
4,LevanteFerries,T0000089,DEN-FOR,Denia,Formentera,2024-01-08 08:00:00,2024-01-08,08:00,0,1,...,872.0,474.0,301.0,122.0,601.20,156.441211,177.30,42.798303,5.75,0.501000
